# ArchVis 직접 추출본 RGB2D 진단
테스트 세션에서 고정한 소스 ZIP을 검증해 Colab 로컬 디렉터리에 푼 뒤 사용합니다. manifest·config·입력 경로와 예산을 확인하고 설정 셀을 채웁니다. 자세한 계약은 저장소의 `docs/ARCHVIS_DIRECT_RGB2D.md`에 있습니다.


In [ ]:
from pathlib import Path

EXECUTE = False
SOURCE_DIR = Path("/content/rewind_source")
EXPECTED_SOURCE_REVISION = ""
CONFIG = Path("/content/archvis_rgb2d_config.json")
EXPECTED_CONFIG_FILE_SHA256 = ""
MANIFEST = Path("/content/archvis_manifest/manifest.json")
MANIFEST_CONTENT_SHA256 = ""
RGB_ROOT = ""
POSE_ROOT = ""
SPEC = SOURCE_DIR / "configs/real_archvis62_v1.json"
OUTPUT = Path("/content/archvis_rgb2d_run")
DEVICE = "cuda"
EPOCHS = 1

# 테스트 세션에서 명시한 값으로 모두 채웁니다.
LIMITS = {
    "max-seconds": None, "max-attempts": None, "max-read-bytes": None,
    "max-rss-bytes": None, "max-disk-bytes": None,
    "max-cuda-allocated-bytes": None, "max-cuda-reserved-bytes": None,
}
EXTRA_ARGS = []  # 고정 재개 또는 전체 확대 증거의 경로와 SHA 인자


In [ ]:
import hashlib
import importlib.util
import re
import subprocess
import sys

if not EXECUTE:
    print("실행 비활성. 테스트 세션의 고정 설정을 확인하세요.")
else:
    assert re.fullmatch(r"[0-9a-f]{40}", EXPECTED_SOURCE_REVISION)
    assert (SOURCE_DIR / "SOURCE_REVISION.txt").read_text().strip() == EXPECTED_SOURCE_REVISION
    assert re.fullmatch(r"[0-9a-f]{64}", EXPECTED_CONFIG_FILE_SHA256)
    assert re.fullmatch(r"[0-9a-f]{64}", MANIFEST_CONTENT_SHA256)
    assert CONFIG.stat().st_size <= 64 * 1024
    assert hashlib.sha256(CONFIG.read_bytes()).hexdigest() == EXPECTED_CONFIG_FILE_SHA256
    assert RGB_ROOT and POSE_ROOT
    assert all(importlib.util.find_spec(m) for m in ("torch", "PIL", "psutil"))
    assert all(type(v) in (int, float) and v > 0 for v in LIMITS.values())
    command = [
        sys.executable, "-m", "rewind_m1.extracted_rgb_cli",
        "--manifest", str(MANIFEST), "--manifest-sha256", MANIFEST_CONTENT_SHA256,
        "--rgb-root", RGB_ROOT, "--pose-root", POSE_ROOT, "--spec", str(SPEC),
        "--config", str(CONFIG), "--output", str(OUTPUT), "--device", DEVICE, "--epochs", str(EPOCHS),
    ]
    for name, value in LIMITS.items():
        command.extend(["--" + name, str(value)])
    result = subprocess.run(command + EXTRA_ARGS, cwd=SOURCE_DIR)
    print("종료 코드", result.returncode, "보고서", OUTPUT / "report.json")
